# Imports

In [1]:
import pandas as pd
import numpy as np 
import re
import string
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
from sklearn.model_selection import train_test_split

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# Exploring data

In [2]:
df=pd.read_csv("/kaggle/input/fake-news-detection-dataset/fake_news_dataset.csv")
df.head()

,title,text,date,source,author,category,label
0,Foreign Democrat final.,more tax development both store agreement lawy...,2023-03-10,NY Times,Paula George,Politics,real
1,To offer down resource great point.,probably guess western behind likely next inve...,2022-05-25,Fox News,Joseph Hill,Politics,fake
2,Himself church myself carry.,them identify forward present success risk sev...,2022-09-01,CNN,Julia Robinson,Business,fake
3,You unit its should.,phone which item yard Republican safe where po...,2023-02-07,Reuters,Mr. David Foster DDS,Science,fake
4,Billion believe employee summer how.,wonder myself fact difficult course forget exa...,2023-04-03,CNN,Austin Walker,Technology,fake


In [3]:
df.drop(columns="date",inplace=True)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   title     20000 non-null  object
 1   text      20000 non-null  object
 2   source    19000 non-null  object
 3   author    19000 non-null  object
 4   category  20000 non-null  object
 5   label     20000 non-null  object
dtypes: object(6)
memory usage: 937.6+ KB


In [5]:
df['source'].nunique()

8

In [6]:
df["has_src"]=df["source"].notna().astype(int)

In [7]:
df.loc[(df['label'] == 'fake') & (df['source'].isnull()), 'source'] = df[df['label'] == 'fake']['source'].mode()[0]
df.loc[(df['label'] == 'real') & (df['source'].isnull()), 'source'] = df[df['label'] == 'real']['source'].mode()[0]

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   title     20000 non-null  object
 1   text      20000 non-null  object
 2   source    20000 non-null  object
 3   author    19000 non-null  object
 4   category  20000 non-null  object
 5   label     20000 non-null  object
 6   has_src   20000 non-null  int64 
dtypes: int64(1), object(6)
memory usage: 1.1+ MB


In [9]:
df['author'].nunique()

17051

In [10]:
df["has_author"]=df["author"].notna().astype(int)
df.drop(columns="author",inplace=True)

In [11]:
df.duplicated().sum()

0

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   title       20000 non-null  object
 1   text        20000 non-null  object
 2   source      20000 non-null  object
 3   category    20000 non-null  object
 4   label       20000 non-null  object
 5   has_src     20000 non-null  int64 
 6   has_author  20000 non-null  int64 
dtypes: int64(2), object(5)
memory usage: 1.1+ MB


In [13]:
df.head()

,title,text,source,category,label,has_src,has_author
0,Foreign Democrat final.,more tax development both store agreement lawy...,NY Times,Politics,real,1,1
1,To offer down resource great point.,probably guess western behind likely next inve...,Fox News,Politics,fake,1,1
2,Himself church myself carry.,them identify forward present success risk sev...,CNN,Business,fake,1,1
3,You unit its should.,phone which item yard Republican safe where po...,Reuters,Science,fake,1,1
4,Billion believe employee summer how.,wonder myself fact difficult course forget exa...,CNN,Technology,fake,1,1


# Pre-Processing

In [14]:
df.nunique()

title         20000
text          20000
source            8
category          7
label             2
has_src           2
has_author        2
dtype: int64

In [15]:
df["label"].replace({"fake":0, "real":1},inplace=True)

/tmp/ipykernel_19/1140773646.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["label"].replace({"fake":0, "real":1},inplace=True)
/tmp/ipykernel_19/1140773646.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["label"].replace({"fake":0, "real":1},inplace=True)


So, now we already have ("label","has_src","has_author") in numerical format has values 0 or 1

let's encode "source" & "category" => Nominal encoding

In [16]:
print (df["source"].unique(),"--------",df["category"].unique())

['NY Times' 'Fox News' 'CNN' 'Reuters' 'Daily News' 'Global Times'
 'The Guardian' 'BBC'] -------- ['Politics' 'Business' 'Science' 'Technology' 'Health' 'Sports'
 'Entertainment']


In [17]:
df = pd.get_dummies(df, columns=['source', 'category'])

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   title                   20000 non-null  object
 1   text                    20000 non-null  object
 2   label                   20000 non-null  int64 
 3   has_src                 20000 non-null  int64 
 4   has_author              20000 non-null  int64 
 5   source_BBC              20000 non-null  bool  
 6   source_CNN              20000 non-null  bool  
 7   source_Daily News       20000 non-null  bool  
 8   source_Fox News         20000 non-null  bool  
 9   source_Global Times     20000 non-null  bool  
 10  source_NY Times         20000 non-null  bool  
 11  source_Reuters          20000 non-null  bool  
 12  source_The Guardian     20000 non-null  bool  
 13  category_Business       20000 non-null  bool  
 14  category_Entertainment  20000 non-null  bool  
 15  ca

In [19]:
ohe_columns = [
    "source_BBC", "source_CNN", "source_Daily News", "source_Fox News",
    "source_Global Times", "source_NY Times", "source_Reuters", "source_The Guardian",
    "category_Business", "category_Entertainment", "category_Health", "category_Politics",
    "category_Science", "category_Sports", "category_Technology"
]
df[ohe_columns] = df[ohe_columns].astype(int)

In [20]:
df.head()

,title,text,label,has_src,has_author,source_BBC,source_CNN,source_Daily News,source_Fox News,source_Global Times,source_NY Times,source_Reuters,source_The Guardian,category_Business,category_Entertainment,category_Health,category_Politics,category_Science,category_Sports,category_Technology
0,Foreign Democrat final.,more tax development both store agreement lawy...,1,1,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0
1,To offer down resource great point.,probably guess western behind likely next inve...,0,1,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0
2,Himself church myself carry.,them identify forward present success risk sev...,0,1,1,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0
3,You unit its should.,phone which item yard Republican safe where po...,0,1,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0
4,Billion believe employee summer how.,wonder myself fact difficult course forget exa...,0,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1


we are done with converting the categorical into numerical 

# Text Cleaning and Word embeddings

In [ ]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.a(r'\s+', ' ', text).strip()
    text = re.sub(r'[^a-z0-9 ]', '', text)
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    
    cleaned_text = ' '.join(tokens)
    return cleaned_text

In [22]:
df["title"]=df["title"].apply(clean_text)
df["text"]=df["text"].apply(clean_text)

In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   title                   20000 non-null  object
 1   text                    20000 non-null  object
 2   label                   20000 non-null  int64 
 3   has_src                 20000 non-null  int64 
 4   has_author              20000 non-null  int64 
 5   source_BBC              20000 non-null  int64 
 6   source_CNN              20000 non-null  int64 
 7   source_Daily News       20000 non-null  int64 
 8   source_Fox News         20000 non-null  int64 
 9   source_Global Times     20000 non-null  int64 
 10  source_NY Times         20000 non-null  int64 
 11  source_Reuters          20000 non-null  int64 
 12  source_The Guardian     20000 non-null  int64 
 13  category_Business       20000 non-null  int64 
 14  category_Entertainment  20000 non-null  int64 
 15  ca

In [24]:
from sentence_transformers import SentenceTransformer

2025-05-30 14:50:38.494886: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748616638.661998      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748616638.711835      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [25]:
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
title_embeddings = sbert_model.encode(df['title'].tolist(), show_progress_bar=True, convert_to_numpy=True)
print(f"Generated {len(title_embeddings)} title embeddings with dimension {title_embeddings.shape[1]}")
text_embeddings = sbert_model.encode(df['text'].tolist(), show_progress_bar=True, convert_to_numpy=True)
print(f"Generated {len(text_embeddings)} text embeddings with dimension {text_embeddings.shape[1]}")

title_embeddings_df = pd.DataFrame(title_embeddings, columns=[f'title_emb_{i}' for i in range(title_embeddings.shape[1])], index=df.index)
text_embeddings_df = pd.DataFrame(text_embeddings, columns=[f'text_emb_{i}' for i in range(text_embeddings.shape[1])], index=df.index)

df = pd.concat([df, title_embeddings_df, text_embeddings_df], axis=1)

df = df.drop(columns=['title', 'text'])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

... (5 outputs left out)

Batches:   0%|          | 0/625 [00:00<?, ?it/s]

Generated 20000 title embeddings with dimension 384


Batches:   0%|          | 0/625 [00:00<?, ?it/s]

Generated 20000 text embeddings with dimension 384


In [26]:
df.shape

(20000, 786)

In [27]:
df.head()

,label,has_src,has_author,source_BBC,source_CNN,source_Daily News,source_Fox News,source_Global Times,source_NY Times,source_Reuters,...,text_emb_374,text_emb_375,text_emb_376,text_emb_377,text_emb_378,text_emb_379,text_emb_380,text_emb_381,text_emb_382,text_emb_383
0,1,1,1,0,0,0,0,0,1,0,...,0.055533,0.095661,-0.012715,0.122311,0.018720,-0.052092,0.091599,-0.077022,-0.057591,0.027619
1,0,1,1,0,0,0,1,0,0,0,...,0.013583,0.085263,-0.020346,0.101425,0.014039,-0.005731,0.009499,-0.083301,-0.010874,0.000984
2,0,1,1,0,1,0,0,0,0,0,...,0.002025,0.072347,0.006825,0.096208,-0.017121,-0.015611,0.099690,-0.052133,-0.038361,0.019182
3,0,1,1,0,0,0,0,0,0,1,...,-0.045016,0.036812,0.005427,0.042037,0.024191,-0.030891,0.097486,-0.053149,-0.005909,-0.021369
4,0,1,1,0,1,0,0,0,0,0,...,0.019599,0.124556,0.044733,0.084491,-0.085495,-0.003436,0.106664,-0.035796,-0.016306,-0.002996


# Data splitting


In [28]:
X = df.drop('label', axis=1)
y = df['label'] 

In [29]:
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)

In [30]:
print(f"Shape of X_train (training features): {X_train.shape}")
print(f"Shape of X_test (testing features): {X_test.shape}")
print(f"Shape of y_train (training target): {y_train.shape}")
print(f"Shape of y_test (testing target): {y_test.shape}")

Shape of X_train (training features): (16000, 785)
Shape of X_test (testing features): (4000, 785)
Shape of y_train (training target): (16000,)
Shape of y_test (testing target): (4000,)


# Model

In [31]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

model = LogisticRegression(max_iter=10000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.51975
              precision    recall  f1-score   support

           0       0.52      0.55      0.54      2011
           1       0.52      0.49      0.50      1989

    accuracy                           0.52      4000
   macro avg       0.52      0.52      0.52      4000
weighted avg       0.52      0.52      0.52      4000



In [32]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))


Random Forest Accuracy: 0.532
              precision    recall  f1-score   support

           0       0.53      0.59      0.56      2011
           1       0.53      0.47      0.50      1989

    accuracy                           0.53      4000
   macro avg       0.53      0.53      0.53      4000
weighted avg       0.53      0.53      0.53      4000



In [33]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [34]:
from sklearn.svm import LinearSVC

svm = LinearSVC(max_iter=10000)
svm.fit(X_train_scaled, y_train)

y_pred_svm = svm.predict(X_test_scaled)

print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))


SVM Accuracy: 0.50725
              precision    recall  f1-score   support

           0       0.51      0.54      0.52      2011
           1       0.50      0.48      0.49      1989

    accuracy                           0.51      4000
   macro avg       0.51      0.51      0.51      4000
weighted avg       0.51      0.51      0.51      4000



/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [35]:
import xgboost as xgb

xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))


XGBoost Accuracy: 0.54675
              precision    recall  f1-score   support

           0       0.55      0.55      0.55      2011
           1       0.54      0.54      0.54      1989

    accuracy                           0.55      4000
   macro avg       0.55      0.55      0.55      4000
weighted avg       0.55      0.55      0.55      4000



In [36]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

model = Sequential([
    Dense(256, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.fit(X_train, y_train, epochs=10, batch_size=64, validation_split=0.1)

loss, acc = model.evaluate(X_test, y_test)
print("Deep Learning Accuracy:", acc)


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1748617157.356020      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13604 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1748617157.356683      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/10


I0000 00:00:1748617160.374910      81 service.cc:148] XLA service 0x7de5b0026100 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1748617160.375520      81 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1748617160.375540      81 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1748617160.622066      81 cuda_dnn.cc:529] Loaded cuDNN version 90300


 71/225 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5141 - loss: 0.6938

I0000 00:00:1748617161.942459      81 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


225/225 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.5229 - loss: 0.6923 - val_accuracy: 0.5375 - val_loss: 0.6835
Epoch 2/10
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5396 - loss: 0.6809 - val_accuracy: 0.5331 - val_loss: 0.6686
Epoch 3/10
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5577 - loss: 0.6660 - val_accuracy: 0.5325 - val_loss: 0.6623
Epoch 4/10
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5741 - loss: 0.6522 - val_accuracy: 0.5288 - val_loss: 0.6571
Epoch 5/10
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6087 - loss: 0.6385 - val_accuracy: 0.5213 - val_loss: 0.6645
Epoch 6/10
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6368 - loss: 0.6187 - val_accuracy: 0.5344 - val_loss: 0.6759
Epoch 7/10
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6611 - loss: 0.5971 - val_accuracy: 0.5325 - val_loss: 0.6785
Epoch 8/10
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6949 - loss: 0.5627 - val_accuracy: 0.5269 - val_

In [37]:
df["label"].value_counts()

label
0    10056
1     9944
Name: count, dtype: int64